# Passive tracers as colored dye

A *passive tracer* is a model variable that moves with the air and does
nothing else. A RAMS run with `ITRACER > 0` carries a stack of them. Each
one marks the air that started in one horizontal layer. At later times, a
tracer's value at a point tells how much of that point's air came from
its layer.

This notebook draws those tracers as a colored dye inside the storm. The
color is where the air came from: its starting height. The result shows
which layers feed the cloud, and where air from aloft has been carried
down.

It covers:

1. finding each tracer's starting height;
2. the dye: how much air is marked, and its color;
3. hiding air that has not moved, so the environment stays clear;
4. a side-by-side figure of the cloud and its dye.

Run it in the pixi `blender` environment. `01_storm_portrait.ipynb`
explains the basics, and `02_winds_at_a_fixed_time.ipynb` explains the
shared camera.

In [ ]:
import numpy as np

import notebook_setup as setup
import skyvista as sv

from cloud_render_config import make_cloud_export_config
from rams_lite_loader import load_rams_lite_hydrometeors
from render_rams_dye_side_by_side import (
    TRACER_INITIAL_MIXING_RATIO,
    tracer_starting_heights,
)
from render_rams_streamlines_side_by_side import pinned_view

## 1. Each tracer's starting height

The tracers are named `TRACERP001`, `TRACERP002`, and so on. RAMS gives
undiluted air from a tracer's layer the value 10^5
(`TRACER_INITIAL_MIXING_RATIO`).

`tracer_starting_heights` reads the run's first output file, at t = 0.
There, each tracer still sits in its layer. The starting height is the
tracer-weighted mean height of that layer.

Tracers that start above the loaded levels (16.5 km) are dropped. Their air
cannot reach those levels without having moved a long way.

In [ ]:
# The loaded levels end at 16.5 km.
TOP_HEIGHT = 16500.0

tracer_names, starting_heights = tracer_starting_heights(setup.RAMS_TRACER_FILE, TOP_HEIGHT)
print(f"{len(tracer_names)} tracers, starting from {starting_heights[0]:.0f} m"
      f" to {starting_heights[-1]:.0f} m")

In [ ]:
# The cloud, and the tracers, at 70 min.
storm_clouds = load_rams_lite_hydrometeors(setup.RAMS_TRACER_FILE, top_height=TOP_HEIGHT)
storm_tracers = load_rams_lite_hydrometeors(
    setup.RAMS_TRACER_FILE, top_height=TOP_HEIGHT, variables=tracer_names
)
storm_minutes = float(storm_clouds["time"]) / 60.0
print(f"t = {storm_minutes:.1f} min")

## 2. One camera for both panels

As in notebook 02, the plain cloud is rendered first with the automatic
camera. The dye panel then reuses that camera and origin shift.

Aerial perspective (haze, see notebook 01) is on in both panels.

This run's lite files do not include the air density `DN0`. The cloud
optics need it to turn mixing ratios (per kg of air) into amounts per
m^3. skyvista therefore uses the 1976 US standard atmosphere instead,
and each render below warns about it. With output that has a density
variable, pass `air_density="<name>"` to `add_clouds` if skyvista does
not find it by itself.

In [ ]:
# Panels are 4:3, so two side by side make a wide figure.
PANEL_RESOLUTION = (800, 600)


def make_preview_config(origin_shift=None):
    """
    Cloud export settings shared by both panels, at preview quality.

    ``make_cloud_export_config`` (examples/rams_clouds) sets the physical
    sky, a low camera south-east of the storm and the sun from the
    north-east. Haze with 80 km visibility is added on top.
    """
    from types import SimpleNamespace

    arguments = SimpleNamespace(
        width=PANEL_RESOLUTION[0], height=PANEL_RESOLUTION[1],
        samples=setup.PREVIEW_SAMPLES,
        sun_elevation=30.0, sun_azimuth=15.0,
        camera_azimuth=-55.0, camera_elevation=-8.0, camera_distance=1.1,
    )
    config = make_cloud_export_config(arguments, origin_shift=origin_shift)
    config.world.aerial_perspective = True
    config.world.visibility_km = 80.0
    return config


# The plain cloud, with the automatic camera.
plain_bundle_dir = setup.WORK_DIR / "dye_plain_cloud"
sv.Scene().add_clouds(storm_clouds, microphysics="rams").to_blender(
    plain_bundle_dir, config=make_preview_config(), render=True, quiet=True
)
setup.show_image(plain_bundle_dir, width=600)
camera_location, camera_look_at, origin_shift = pinned_view(plain_bundle_dir)


def make_pinned_config():
    """Preview settings with the plain cloud's camera and origin."""
    config = make_preview_config(origin_shift=origin_shift)
    config.camera.location = camera_location
    config.camera.look_at = camera_look_at
    config.overlay_colorbars = False
    return config

## 3. The dye

`add_dye` turns the tracers into one colored volume. At each grid point:

- the *dyed fraction* is the sum of all tracers divided by
  `reference_value`. It is the share of the air that any tracer marks, from
  0 to 1.
- the *label* is the tracer-weighted mean of `tracer_values`. With starting
  heights as values, the label is the mean starting height of the air.

Blender renders the dye as a medium that scatters and absorbs light in the
colormap color of the label. Its density follows the dyed fraction.

- `extinction_per_km`: the optical depth of 1 km of undiluted dye. Larger
  values make the dye more opaque.
- `self_illumination`: a glow, as a fraction of display white. It keeps
  the color visible in the cloud's shadow.

As in notebook 02, the cloud is thinned to 3% of its extinction so the dye
inside shows.

The first render below dyes all the marked air. It shows why section 4 is
needed.

In [ ]:
# Starting heights at the ends of the colormap (m).
HEIGHT_LIMITS = (0.0, 3000.0)


def render_dye_panel(bundle_name, min_vertical_displacement):
    """Render the thinned cloud plus the dye; return the bundle directory."""
    bundle_dir = setup.WORK_DIR / bundle_name
    dye_scene = sv.Scene().add_clouds(
        storm_clouds, microphysics="rams", extinction_scale=0.03
    )
    dye_scene.add_dye(
        storm_tracers,
        tracers=tracer_names,
        tracer_values=starting_heights,
        reference_value=TRACER_INITIAL_MIXING_RATIO,
        min_vertical_displacement=min_vertical_displacement,
        cmap="turbo",
        clim=HEIGHT_LIMITS,
        extinction_per_km=3.0,
        self_illumination=0.3,
    )
    dye_scene.to_blender(bundle_dir, config=make_pinned_config(), render=True, quiet=True)
    return bundle_dir


all_dye_bundle_dir = render_dye_panel("dye_all_marked_air", min_vertical_displacement=None)
setup.show_image(all_dye_bundle_dir, width=600)

## 4. Hide the air that has not moved

In the render above, every tracer layer is still mostly where it started.
The whole lower atmosphere is dye, and the storm is lost inside it.

`min_vertical_displacement` keeps only air that has moved more than a set
distance from its starting height. Here that distance is 1 km. Below about
1 km, turbulence mixes the boundary layer through its depth, so its
tracers would show everywhere.

What remains is air that the storm has moved: boundary-layer air lifted in
the updraft, and air from aloft brought down.

In [ ]:
moved_dye_bundle_dir = render_dye_panel("dye_moved_air", min_vertical_displacement=1000.0)
setup.show_image(moved_dye_bundle_dir, width=600)

## 5. Side by side

The colorbar labels starting heights in kilometers. `tick_scale=0.001`
converts the tick values from meters for display only.

In [ ]:
# Text heights, as fractions of the panel height.
TITLE_SIZE, LABEL_SIZE = 0.04, 0.03

left_panel = sv.compose.Panel(
    setup.rendered_frame(plain_bundle_dir),
    [
        sv.compose.Text("Cloud, physically based", size=TITLE_SIZE),
        sv.compose.Text(
            f"t = {storm_minutes:.1f} min", size=LABEL_SIZE, offset=(0.0, -0.055)
        ),
    ],
)
right_panel = sv.compose.Panel(
    setup.rendered_frame(moved_dye_bundle_dir),
    [
        sv.compose.Text("Passive tracers as dye", size=TITLE_SIZE),
        sv.compose.Text(
            "Dyed: air moved > 1 km vertically\nCloud extinction x 0.03",
            anchor="lower left",
            size=LABEL_SIZE,
        ),
        sv.compose.Colorbar(
            cmap="turbo",
            clim=HEIGHT_LIMITS,
            label="Starting height of the air (km)",
            ticks=list(np.linspace(*HEIGHT_LIMITS, 4)),
            tick_scale=0.001,
            anchor="lower right",
            length=0.35,
            text_size=LABEL_SIZE,
        ),
    ],
)
figure_path = setup.WORK_DIR / "dye_side_by_side.png"
sv.compose.Figure(panels=[left_panel, right_panel]).render(figure_path)
setup.show_image(figure_path, width=1000)

## Where to go from here

`examples/rams_clouds/render_rams_dye_side_by_side.py` makes this figure
from the command line. Its options set the displacement threshold, the dye
opacity, the glow and the colormap.

Dye works with any set of tracers. Without `tracer_values`, tracers are
labelled by their index. With a single tracer, the label is constant and
only the dyed fraction varies.